In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Visualisasi pemasaan Circuit

<Accordion>
<AccordionItem title="Versi pakej">

Kod di halaman ini dibangunkan menggunakan keperluan berikut.
Kami syorkan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Walaupun [pelukis garis masa](/guides/visualize-circuit-timing) yang terbina dalam Qiskit berguna untuk Circuit statik, ia mungkin tidak mencerminkan pemasaan [Circuit dinamik](/guides/classical-feedforward-and-control-flow) dengan tepat kerana operasi implisit seperti penyiaran dan penentuan cabang. Sebagai sebahagian daripada sokongan Circuit dinamik, Qiskit Runtime mengembalikan maklumat pemasaan Circuit yang tepat di dalam keputusan kerja apabila diminta.

> **Note:** - Ini adalah fungsi eksperimen. Ia berada dalam status keluaran pratonton dan oleh itu tertakluk kepada perubahan.
> - Fungsi ini hanya terpakai pada kerja Sampler Qiskit Runtime.
> - Walaupun jumlah masa Circuit dikembalikan dalam metadata "compilation", ini BUKAN masa yang digunakan untuk pengebilan (masa QPU).
### Aktifkan pengambilan data pemasaan
Untuk mengaktifkan pengambilan data pemasaan, tetapkan bendera eksperimen `scheduler_timing` kepada `True` apabila menjalankan kerja primitif.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Akses data pemasaan Circuit
Apabila diminta, data pemasaan Circuit untuk setiap PUB dikembalikan dalam metadata keputusan kerja, di bawah `["compilation"]["scheduler_timing"]["timing"]`. Medan ini mengandungi maklumat pemasaan mentah. Untuk memaparkan maklumat pemasaan, gunakan alat visualisasi terbina dalam, seperti yang diterangkan dalam bahagian [Visualisasi pemasaan](#visualize-timings).

Gunakan kod berikut untuk mengakses data pemasaan Circuit bagi PUB pertama:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Fahami data pemasaan mentah
Walaupun memvisualisasi data pemasaan Circuit menggunakan kaedah `draw_circuit_schedule_timing` adalah kes penggunaan paling biasa, mungkin berguna untuk memahami struktur data pemasaan mentah yang dikembalikan. Ini boleh membantu anda, misalnya, untuk mengekstrak maklumat secara aturcara.

Data pemasaan yang dikembalikan dalam `["compilation"]["scheduler_timing"]["timing"]` adalah senarai rentetan. Setiap rentetan mewakili satu arahan pada saluran tertentu dan dipisahkan koma kepada jenis data berikut:

- `Branch` - Menentukan sama ada arahan berada dalam aliran kawalan (then / else) atau cabang utama.
- `Instruction` - Gate dan Qubit yang perlu dioperasikan.
- `Channel` - Saluran yang ditetapkan dengan arahan. Ia boleh menjadi salah satu daripada berikut:
      - `Qubit x` - Saluran pemacu untuk Qubit _x_.
      - `AWGRx_y` (penjana gelombang sewenang-wenang readout) - Digunakan oleh saluran readout untuk berkomunikasi apabila mengukur Qubit. Argumen _x_ dan _y_ sepadan dengan ID instrumen readout dan nombor Qubit, masing-masing.
- `T0` - Masa mula arahan dalam jadual lengkap
- `Duration` - Tempoh arahan, dalam unit _dt_ saat, di mana 1 dt = 1 kitaran penjadualan. Anda boleh mencari nilai `dt` bagi Backend menggunakan [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Jenis operasi nadi yang digunakan.

Contoh:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Visualisasi pemasaan
Dengan `qiskit-ibm-runtime` v0.43.0 atau lebih baru, anda boleh memvisualisasi pemasaan Circuit. Untuk memvisualisasi pemasaan, anda perlu menukar metadata keputusan kepada `fig` menggunakan [kaedah `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). Kaedah ini mengembalikan rajah `plotly`, yang boleh anda paparkan terus, simpan ke fail, atau kedua-duanya. Untuk maklumat lanjut tentang arahan `plotly` yang perlu digunakan, lihat [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) dan [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Hovering over the output shows information such as the start, finish, and duration.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Example of a generated figure')
#### Fahami rajah yang dihasilkan
Imej data pemasaan Circuit yang dihasilkan oleh `draw_circuit_schedule_timing` menyampaikan maklumat berikut:

- Paksi X adalah masa dalam unit _dt_ saat, di mana 1 dt = 1 kitaran penjadualan. Anda boleh mencari nilai `dt` bagi Backend menggunakan [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Paksi Y adalah saluran (anggap saluran sebagai instrumen yang memancarkan nadi).
    - `Receive channel` - Satu-satunya saluran yang bukan instrumen itu sendiri. Ia adalah arahan yang dimainkan pada semua saluran yang merupakan sebahagian daripada prosedur komunikasi dengan hab pada masa itu.
    - `Qubit x` - Saluran pemacu untuk Qubit x.
    - `AWGRx_y` (penjana gelombang sewenang-wenang readout) - Digunakan oleh saluran readout untuk berkomunikasi apabila mengukur Qubit. Argumen _x_ dan _y_ sepadan dengan ID instrumen readout dan nombor Qubit, masing-masing.
    - `Hub` - Mengawal penyiaran.

Selain itu, setiap arahan mempunyai format *X_Y*, di mana *X* adalah nama arahan dan *Y* adalah jenis nadi. Jenis `play` menerapkan nadi kawalan, dan `capture` merekod keadaan Qubit. Anda juga boleh mengarah kursor ke atas setiap arahan untuk mendapatkan butiran lanjut. Sebagai contoh, rajah sebelumnya menunjukkan nadi kawalan untuk Gate X yang diterapkan pada Qubit 10 pada 1161 dt.
### Contoh hujung ke hujung
Contoh ini menunjukkan cara mengaktifkan pilihan, mendapatkan maklumat pemasaan Circuit dari metadata, dan memaparkannya sebagai imej.

Pertama, sediakan persekitaran, tentukan Circuit dan tukarkannya kepada Circuit ISA, dan tentukan serta jalankan kerja.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Seterusnya, dapatkan pemasaan jadual Circuit: